In [1]:
import os, sys

# 実行環境に応じてプロジェクトルートを決めて移動する
try:
    # .py として実行された場合は __file__ からルートを解決
    PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), '..'))
except NameError:
    # ノートブック実行(__file__ が無い)
    if 'google.colab' in sys.modules:
        # Colab: Drive をマウントしてプロジェクトを配置した場所へ
        from google.colab import drive
        drive.mount('/content/drive')
        PROJECT_ROOT = '/content/drive/MyDrive/deep-learning-6'  # アップロード先に合わせる
    else:
        # ローカルの Jupyter/VSCode: codebot が見つかるまで親をたどる
        PROJECT_ROOT = os.getcwd()
        while not os.path.isdir(os.path.join(PROJECT_ROOT, 'codebot')):
            parent = os.path.dirname(PROJECT_ROOT)
            if parent == PROJECT_ROOT:
                raise RuntimeError('プロジェクトルート(codebotを含む)が見つかりません')
            PROJECT_ROOT = parent

os.chdir(PROJECT_ROOT)
sys.path.append(PROJECT_ROOT)

Mounted at /content/drive


In [2]:
import torch
import torch.nn.functional as F
from codebot.model import GPT
from codebot.tokenizer import BPETokenizer
from codebot.utils import get_device

### setting

In [3]:
device = get_device()
model_path = "codebot/model_pretrain.pt"
tokenizer_path = "codebot/merge_rules.pkl"

### 生成設定

In [11]:
prompt = "class"
max_new_tokens = 200
temperature = 1.0

In [8]:
@torch.no_grad()
def generate(model, tokenizer, prompt, max_new_tokens=1000, temperature=1.0):
    model.eval()

    device = next(model.parameters()).device
    ids = tokenizer.encode(prompt)
    ids = torch.tensor([ids], dtype=torch.long, device=device)

    generated_ids = ids.clone()

    # トークン生成ループ
    for _ in range(max_new_tokens):
        # context長を超えた場合は古いトークンを切り捨てる
        if ids.size(1) > model.max_context_len:
            ids = ids[:, -model.max_context_len:]

        # 最後の位置のロジットを取得（次トークンの予測）
        logits = model(ids)[:, -1, :]
        if temperature == 0:
            next_id = logits.argmax(dim=-1, keepdim=True)
        else:
            probs = F.softmax(logits / temperature, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)

        # 終了トークンが生成されたら停止
        if next_id.item() == tokenizer.end_token_id:
            break

        # 生成したトークンを追加
        ids = torch.cat((ids, next_id), dim=1)
        generated_ids = torch.cat((generated_ids, next_id), dim=1)

    # decodeして返す
    generated_text = tokenizer.decode(generated_ids[0].tolist())
    return generated_text

In [9]:
tokenizer = BPETokenizer.load_from(tokenizer_path)
model = GPT.load_from(model_path, device=device)

In [12]:
# テキスト生成
for i in range(5):
    print(f"--- サンプル {i+1} ---")
    generated_text = generate(
        model=model,
        tokenizer=tokenizer,
        prompt=prompt,
        max_new_tokens=max_new_tokens,
        temperature=temperature
    )
    print(generated_text)
    print()

--- サンプル 1 ---
class Caler.MyCaler()
    X = X[features].reshape(-1,1)
    #Calculate accuracy split
    accuracy = scaler.fit_transform(X)

    #Calculate accuracy
    accuracy = np.mean(accuracy)

    ## classify
    intercept = tf.argmax()

    if intercept[1]:
        content = content[0].min()

    if intent == prediction:
        print("Catenated matrix")
    else:
        print("Invalid matrix")


--- サンプル 2 ---
class Eurlopment('ContentProvideo.xml')
customers = []

class Post(Resource):
    def __init__(self, pople):
        # initialize a dictionary
        pass

    def post(self, pople):
        # create a new post
        post = Post("[Posttd]", port)
        # create new post with customers
        post.create_post(post)


--- サンプル 3 ---
class = 'level-male-male-male-up-male-model'

# Get the data from the URL
data = requests.get(url).json()

# Print the data
for item in data:
 print(item.text)


--- サンプル 4 ---
class CurrentCurrentCurrentCurrentCurrentCourses(EurrentCurre